In [ ]:
import json
import numpy as np
import pandas as pd
from google.colab import files
from sklearn.ensemble import RandomForestClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import classification_report, confusion_matrix, f1_score
from sklearn.model_selection import cross_val_score, train_test_split
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler

# 1. Cargar dataset maestro enriquecido del Producto 1
df_base = pd.read_csv("dataset_inteligencia_territorial.csv", sep=None, engine="python")
df_base.columns = [str(c).strip().lower() for c in df_base.columns]

col_lum = next((c for c in df_base.columns if "lum" in c), "luminosidad_media_qc")
df_base["luminosidad"] = pd.to_numeric(df_base[col_lum].astype(str).str.replace(",", "."), errors="coerce").fillna(75.43)
df_base["centro_lat"] = pd.to_numeric(df_base["centro_lat"].astype(str).str.replace(",", "."), errors="coerce")
df_base["centro_lon"] = pd.to_numeric(df_base["centro_lon"].astype(str).str.replace(",", "."), errors="coerce")
df_base["nivel_base"] = df_base["nivel_incidencia"].astype(str).str.strip()

mapa_clases = {
    "Sin registros nocturnos": "Bajo",
    "Bajo": "Bajo",
    "Medio": "Medio",
    "Alto": "Alto",
}
df_base["nivel_base"] = df_base["nivel_base"].map(mapa_clases).fillna("Bajo")
df_base = df_base.dropna(subset=["centro_lat", "centro_lon"]).copy()

# 2. Expandir con contexto temporal nocturno
franjas_nocturnas = [20, 21, 22, 23, 0, 1, 2, 3, 4, 5, 6]
registros = []

for _, row in df_base.iterrows():
    for hora in franjas_nocturnas:
        for es_fds in [0, 1]:
            target = row["nivel_base"]
            if hora in [0, 1, 2, 3, 4] and es_fds == 1 and target == "Medio":
                target = "Alto"
            elif hora in [22, 23, 0, 1, 2] and es_fds == 1 and target == "Bajo" and row["luminosidad"] < 60:
                target = "Medio"

            registros.append({
                "centro_lat": row["centro_lat"],
                "centro_lon": row["centro_lon"],
                "luminosidad": row["luminosidad"],
                "franja_horaria": hora,
                "es_fin_de_semana": es_fds,
                "nivel_incidencia_objetivo": target,
            })

df_ml = pd.DataFrame(registros)

FEATURES = ["centro_lat", "centro_lon", "luminosidad", "franja_horaria", "es_fin_de_semana"]
TARGET = "nivel_incidencia_objetivo"

X = df_ml[FEATURES]
y = df_ml[TARGET]

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.20, random_state=42, stratify=y
)

# 3. Evaluación comparativa (LogisticRegression vs RandomForest)
log_pipe = Pipeline([("scaler", StandardScaler()), ("clf", LogisticRegression(max_iter=1000, random_state=42))])
rf_model = RandomForestClassifier(n_estimators=40, max_depth=8, random_state=42)

for nombre, mod in [("LogisticRegression", log_pipe), ("RandomForest", rf_model)]:
    cv = cross_val_score(mod, X_train, y_train, cv=5, scoring="f1_macro")
    mod.fit(X_train, y_train)
    preds = mod.predict(X_test)
    print(f"\n=== {nombre} ===")
    print(f"CV F1-Macro: {cv.mean():.4f} | Test F1-Macro: {f1_score(y_test, preds, average='macro'):.4f}")
    print(classification_report(y_test, preds))

# 4. Serializar árboles del RandomForest a JSON puro (sin pickle, 100% portable entre versiones)
arboles_json = []
for est in rf_model.estimators_:
    t = est.tree_
    arboles_json.append({
        "children_left": t.children_left.tolist(),
        "children_right": t.children_right.tolist(),
        "feature": t.feature.tolist(),
        "threshold": np.round(t.threshold, 6).tolist(),
        "value": (t.value[:, 0, :] / t.value[:, 0, :].sum(axis=1, keepdims=True)).round(4).tolist(),
    })

artefacto_json = {
    "modelo_nombre": "RandomForest",
    "features": FEATURES,
    "clases": list(rf_model.classes_),
    "metrica_f1_macro": round(float(f1_score(y_test, rf_model.predict(X_test), average="macro")), 4),
    "arboles": arboles_json,
}

with open("modelo_riesgo_caba.json", "w", encoding="utf-8") as f:
    json.dump(artefacto_json, f)

files.download("modelo_riesgo_caba.json")


=== LogisticRegression ===
CV F1-Macro: 0.2752 | Test F1-Macro: 0.2802
              precision    recall  f1-score   support

        Alto       0.00      0.00      0.00      1457
        Bajo       0.62      0.98      0.76      4377
       Medio       0.40      0.04      0.08      1206

    accuracy                           0.62      7040
   macro avg       0.34      0.34      0.28      7040
weighted avg       0.45      0.62      0.49      7040



/usr/local/lib/python3.13/dist-packages/sklearn/metrics/_classification.py:1565: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", len(result))
/usr/local/lib/python3.13/dist-packages/sklearn/metrics/_classification.py:1565: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", len(result))
/usr/local/lib/python3.13/dist-packages/sklearn/metrics/_classification.py:1565: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", len(result))



=== RandomForest ===
CV F1-Macro: 0.8306 | Test F1-Macro: 0.8224
              precision    recall  f1-score   support

        Alto       0.76      0.89      0.82      1457
        Bajo       0.95      0.92      0.94      4377
       Medio       0.74      0.67      0.71      1206

    accuracy                           0.87      7040
   macro avg       0.82      0.83      0.82      7040
weighted avg       0.88      0.87      0.87      7040



<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>